# ReconShield — Notebook CRISP-DM

**Tugas Akhir:** *Penerapan Algoritma Random Forest dan Dynamic Tarpit dalam Mitigasi Serangan Reconnaissance*

Notebook ini adalah artefak dokumentasi tahap **CRISP-DM** untuk laporan TA:

1. **Business Understanding** — tujuan & metrik yang dipakai
2. **Data Understanding** — karakteristik dataset NetFlow lab
3. **Data Preparation** — labeling, pembuangan IP/noise, split
4. **Modeling** — Random Forest (parameter = `reconshield-ml/config.py`)
5. **Evaluation** — metrik yang sama persis dengan `metrics.json` di dashboard
6. **Deployment** — jalur dari training ke API → web

**Sumber data:** `dataset/ta-recon/data/dataset.csv` (hasil `netflow_v5_collector.py` → `build_dataset.py`, 6.404 flow nyata dari testbed).

> Jalankan dari folder mana pun dalam repo (path dataset dicari otomatis di cell pertama).
> Di Google Colab: upload `dataset.csv` dulu, lalu set `DATA_PATH` secara manual bila diperlukan.

## 1. Business Understanding

**Tujuan:** mengklasifikasi tiap flow NetFlow menjadi `Normal` / `Reconnaissance` sehingga pipeline ReconShield dapat memberi keputusan otomatis (deteksi → aturan Dynamic Tarpit) tanpa pemicu manual.

**Batasan (proposal §3):** port scan L3/L4 (Nmap, Hping3); NetFlow out-of-band bawaan MikroTik; algoritma hanya Random Forest; testbed lab.

**Metrik fokus: precision, recall, dan F1 kelas `Reconnaissance`** — bukan akurasi semata, karena:

- kelas tidak seimbang (normal lebih banyak),
- *false positive* berarti trafik sah ikut ditarpit, *false negative* berarti scan lolos,
- angka akurasi bisa terkecoh oleh distribusi kelas.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

CANDIDATES = [
    Path("../dataset/ta-recon/data/dataset.csv"),     # cwd = reconshield-ml/
    Path("../../dataset/ta-recon/data/dataset.csv"),  # cwd = reconshield-ml/notebooks/
    Path("dataset/ta-recon/data/dataset.csv"),        # cwd = root repo
    Path("dataset.csv"),                              # file upload manual / Colab
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_PATH is not None, "dataset.csv tidak ditemukan - sesuaikan CANDIDATES / upload dulu"
print("dataset:", DATA_PATH.resolve())

## 2. Data Understanding

Rantai pengumpulan data (lab nyata, lihat `dataset/ta-recon/` dan `docs/testbed.md`):

```
MikroTik hAP lite (RouterOS 6.48.6)
  |- NetFlow v5 export (UDP 2055) --> netflow_v5_collector.py --> flows.csv (9.573 flow)
  |- Syslog firewall [RECON-SCAN] --> syslog_collector.py      --> firewall_parsed.csv
                                          |
                              build_dataset.py (labeling IP + fitur)
                                          v
                                    dataset.csv (6.404 baris)
```

- **Label ground truth** (bukan hasil deteksi): `recon` = flow `attacker-net <-> victim-net`, `normal` = sisanya.
- **Noise dibuang**: port 2055 (export NetFlow), 514 (syslog), 8291 (Winbox).
- **16 fitur per-flow**; kolom IP (`src`, `dst`, `exporter`) sengaja **tidak** dipakai sebagai fitur agar model tidak menghafal subnet.

In [ ]:
df = pd.read_csv(DATA_PATH)
print("baris x kolom:", df.shape)
print("kolom:", list(df.columns))
print()
print(df.head(3).to_string(index=False))

In [ ]:
print("distribusi label:")
print(df["label"].value_counts().to_string())
print("rasio: %.1f%% normal / %.1f%% recon" % tuple(df["label"].value_counts(normalize=True) * 100))

missing = df.isna().sum()
missing = missing[missing > 0]
print()
print("missing values:", "tidak ada" if missing.empty else missing.to_dict())
print("duplikat baris :", int(df.duplicated().sum()))

In [ ]:
# Karakteristik per kelas - inti "data understanding" untuk laporan
print("rata-rata fitur kuantitatif per kelas:")
print(df.groupby("label")[["packets", "bytes", "duration_ms", "bytes_per_pkt", "pkts_per_sec"]]
      .mean().round(2).to_string())

print()
print("rata-rata flag TCP per kelas:")
print(df.groupby("label")[["f_syn", "f_ack", "f_rst", "f_psh"]].mean().round(3).to_string())

print()
print("protokol (6=TCP, 17=UDP) per kelas:")
print(pd.crosstab(df["label"], df["prot"]).to_string())

## 3. Data Preparation

- **Encoding label** ke kanonik `Normal` / `Reconnaissance` (alias yang sama dengan `reconshield-ml/training/train_rf.py`, jadi hasilnya identik dengan yang dibaca dashboard).
- **Fitur** = 16 kolom persis `preprocess/features.py::FEATURE_COLUMNS` (kontrak training & inference).
- **Split stratified 80/20**, `random_state=42` — sama dengan `RF_TEST_SIZE` & `RF_RANDOM_STATE` di `config.py` supaya angka di notebook ini **reproduksi `metrics.json`**.

In [ ]:
FEATURES = ["prot", "tos", "tcp_flags", "sport", "dport", "packets", "bytes",
            "duration_ms", "bytes_per_pkt", "pkts_per_sec",
            "f_fin", "f_syn", "f_rst", "f_psh", "f_ack", "f_urg"]

LABEL_MAP = {"normal": "Normal", "benign": "Normal", "legit": "Normal",
             "recon": "Reconnaissance", "reconnaissance": "Reconnaissance",
             "scan": "Reconnaissance", "attack": "Reconnaissance"}

y = df["label"].astype(str).str.strip().str.lower().map(LABEL_MAP)
assert y.notna().all(), "ada label yang tidak dikenal"
X = df[FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0.0)

TEST_SIZE, RANDOM_STATE = 0.2, 42   # = reconshield-ml/config.py
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

print(f"{len(X_train)} train / {len(X_test)} test baris, {X.shape[1]} fitur")
print("kelas:", y.value_counts().to_dict())

## 4. Modeling

Random Forest Classifier dengan parameter yang **sama dengan `training/train_rf.py`** (`build_model()`):

| Parameter | Nilai | Sumber |
|---|---|---|
| `n_estimators` | 100 | `RF_N_ESTIMATORS` |
| `max_depth` | 12 | `RF_MAX_DEPTH` |
| `random_state` | 42 | `RF_RANDOM_STATE` |
| `class_weight` | default (None) | mengikuti service |

> Catatan: `dataset/ta-recon/train_rf.py` memakai 200 pohon + `class_weight=balanced`.
> Angka dashboard memakai konfigurasi service; bila mau dibandingkan, ubah parameter di sini lalu jalankan ulang.

In [ ]:
model = RandomForestClassifier(
    n_estimators=100, max_depth=12, random_state=42, n_jobs=-1,
)
model.fit(X_train, y_train)
print(f"model siap: {model.n_estimators} pohon, max_depth={model.max_depth}")

## 5. Evaluation

Metrik yang dihitung identik dengan `train_rf.py::evaluate()`:

- Accuracy / Precision / Recall / F1 dalam **persen (0–100)**, pos label = `Reconnaissance`
- Confusion matrix dengan urutan `[Normal, Recon]` -> `trueNormal`, `falseThreat`, `missedThreat`, `trueThreat`
- Feature importance ternormalisasi (sum = 1, skala Gini)

In [ ]:
y_pred = model.predict(X_test)
labels = ["Normal", "Reconnaissance"]
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=labels).ravel()

accuracy  = round(float(accuracy_score(y_test, y_pred)) * 100, 2)
precision = round(float(precision_score(y_test, y_pred, pos_label="Reconnaissance")) * 100, 2)
recall    = round(float(recall_score(y_test, y_pred, pos_label="Reconnaissance")) * 100, 2)
f1        = round(float(f1_score(y_test, y_pred, pos_label="Reconnaissance")) * 100, 2)

print(f"Accuracy : {accuracy}%")
print(f"Precision: {precision}%")
print(f"Recall   : {recall}%")
print(f"F1       : {f1}%")
print()
print(f"Confusion matrix [Normal, Recon]: "
      f"trueNormal={tn} falseThreat={fp} missedThreat={fn} trueThreat={tp}")
print()
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
imp = sorted(zip(FEATURES, model.feature_importances_), key=lambda t: t[1], reverse=True)
for name, val in imp:
    print(f"{name:>16}: {val:.4f} " + "#" * int(val * 100))

try:
    import matplotlib.pyplot as plt
    names = [n for n, _ in imp][::-1]
    vals  = [v for _, v in imp][::-1]
    plt.figure(figsize=(8, 5))
    plt.barh(names, vals, color="#4A90E2")
    plt.xlabel("Gini importance")
    plt.title("Feature Importance - Random Forest ReconShield")
    plt.tight_layout()
    plt.show()
except ImportError:
    print()
    print("(matplotlib tidak tersedia - tabel di atas tetap lengkap untuk laporan)")

In [ ]:
# Cross-validation: pengecekan apakah skor test-set realistis
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")
print(f"CV F1-macro (5-fold): {scores.mean():.4f} +/- {scores.std():.4f}")
print("per-fold:", [round(float(s), 4) for s in scores])

## 6. Hasil & Pembahasan (isi setelah run)

Templat yang disarankan untuk Bab Evaluasi laporan:

- **Test-set**: accuracy/precision/recall/F1 = ______ ; confusion matrix = ______.
- **Cross-validation**: F1-macro = ______ ± ______.
- Jika test-set nyaris sempurna **tetapi** CV lebih rendah → dua hal sekaligus:
  1. kelas memang mudah dipisahkan (recon = paket kecil ~44–52 B, 1 paket, SYN murni; normal = payload besar & flow panjang), **dan**
  2. ada kemungkinan **kebocoran**: split acak pada data yang berkorelasi waktu membuat flow dari skenario/sesi yang sama tersebar ke train & test.
- **Rekomendasi bila diperlukan evaluasi lebih ketat:** `GroupShuffleSplit` per-skenario capture (kelompokkan capture per sesi serangan/baseline).
- Laporkan apa adanya — angka 100% tanpa pembahasan keterbatasan justru berisiko ditanya penguji.

**Kontrak hasil:** angka yang tampil di dashboard ReconShield berasal dari
`reconshield-ml/data/processed/metrics.json` — bandingkan kedua angka; harus identik
bila dataset & parameter sama.

## 7. Deployment (tahap akhir CRISP-DM)

Jalur dari notebook ke dashboard:

```bash
# 1. Training resmi (menulis model + metrics.json - kontrak web)
cd reconshield-ml
python -m training.train_rf --data ../dataset/ta-recon/data/dataset.csv --target label --version v1.1.0

# 2. Sajikan API inference
uvicorn api.server:app --port 8000

# 3. (Opsional) publikasikan metrik ke PostgreSQL agar dashboard baca source=database
python -m training.sync_metrics_db
```

Catatan Colab:

- Pin versi `scikit-learn` **sama** dengan mesin lokal saat menyimpan/memuat `.joblib`.
- Unduh `models/*.joblib` + `data/processed/metrics.json` kembali ke lokal.
- Kontrak fitur ada di **satu tempat**: `reconshield-ml/preprocess/features.py` (16 fitur).
  Bila kolom berubah, ubah di sana + pipeline `dataset/ta-recon/build_dataset.py` bersamaan, lalu training ulang.